# 11 — Conversation Memory & Context Quantization (`memory_k` / `summary_k`)

`Convo` memory keeps every prior trial in one continuous conversation (see
[Memory Types](../guides/memory_types.md)). Left unbounded, that conversation
grows one exchange per trial forever. Two `ExpCard` fields quantize it:

- **`memory_k`** — a hard window. Only the last `memory_k` messages are kept;
  everything older is dropped.
- **`summary_k`** — instead of dropping the overflow, fold it into a rolling
  summary (prepended to the system message) once the overflow reaches
  `summary_k` messages.

This notebook runs the same 6-item survey under four memory configurations
and inspects, trial by trial, how many messages the model actually sees and
whether a summary has been produced — using `mock-chat-model` so it runs
with no API key.

In [1]:
from pathlib import Path
import shutil

from psychscanner import ExpCardInit, ExpCard, ScannerModel

RUN_DIR = Path.cwd() / "_memory_quant_demo_runs"
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

task_file = Path.cwd() / "tasks" / "example_survey.json"


## Helper

Every run below shares the same task (`tasks/example_survey.json`, 6 trials)
and the same mock model, varying only `memory`, `memory_k`, `summary_k`, and
`chain_type`. `ScannerModel.run()` returns one dict per trial; `pred_dict`
is the live LangGraph state *after* that trial, so `pred_dict["inputs"]`
is the exact message list the model would see next, and
`pred_dict.get("summary")` is the current rolling summary (if any).

In [2]:
def run_config(label, memory, memory_k, summary_k, chain_type="task"):
    card = ExpCardInit()
    card.model        = "mock-chat-model"
    card.family       = "mock-llm"
    card.memory       = memory
    card.memory_k     = memory_k
    card.summary_k    = summary_k
    card.chain_type   = chain_type
    card.task_file    = task_file
    card.task_context = True
    card.cogtype      = "no"
    card.nsim         = 1              # one simulated participant, one conversation
    card.parser       = "0"
    card.tunnel_status = "0"
    card.proj_dir     = RUN_DIR
    card.projectname  = label
    card.enabletqdm   = False

    exp = ExpCard(card)
    results = ScannerModel(expcard=exp).run(save_str="demo")

    print(f"{label}  (memory={memory}, memory_k={memory_k}, summary_k={summary_k})")
    print(f"{'trial':<6}{'trcode':<8}{'messages':<10}summary")
    for trial in results[0]:
        pred = trial["pred_dict"]
        n_msgs = len(pred["inputs"])
        summary = pred.get("summary") or ""
        print(f"{trial['trial_idx']:<6}{trial['trcode']:<8}{n_msgs:<10}{summary[:50]!r}")
    return results


## A. Baseline — `SingleTurn`

Every trial is a fresh conversation: the model never sees more than the
current stimulus + response, regardless of trial count.

In [3]:
_ = run_config("singleturn", memory="SingleTurn", memory_k=-1, summary_k=0, chain_type="item")

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs/singleturn/example_survey/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [00:00, 308.64it/s]


2026-07-06 11:03:03.840 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:03:03.842 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


singleturn  (memory=SingleTurn, memory_k=-1, summary_k=0)
trial trcode  messages  summary
0     O_1     2         ''
1     O_2     2         ''
2     O_3     2         ''
3     C_1     2         ''
4     C_2     2         ''
5     C_3     2         ''


## B. `Convo`, unlimited history (`memory_k=-1`, default)

All 6 trials share one thread. Message count grows by 2 every trial
(one human stimulus + one AI response) with nothing dropped — by trial 5
the model is re-reading the entire session.

In [4]:
_ = run_config("convo_unlimited", memory="Convo", memory_k=-1, summary_k=0)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs/convo_unlimited/example_survey/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [00:00, 263.80it/s]


2026-07-06 11:03:03.893 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:03:03.895 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


convo_unlimited  (memory=Convo, memory_k=-1, summary_k=0)
trial trcode  messages  summary
0     O_1     2         ''
1     O_2     4         ''
2     O_3     6         ''
3     C_1     8         ''
4     C_2     10        ''
5     C_3     12        ''


## C. Quantizing with `memory_k` — hard truncation window

With `memory_k=4`, once the conversation exceeds 4 messages the oldest
overflow is dropped before each new call. Message count stabilizes instead
of growing — it settles at `memory_k + 1` rather than exactly `memory_k`,
because trimming runs on the *incoming* state (previous turns + the new
stimulus) before this trial's response is appended.

`summary_k=0` here means dropped messages are gone for good — no summary is
produced.

In [5]:
_ = run_config("convo_k4", memory="Convo", memory_k=4, summary_k=0)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs/convo_k4/example_survey/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [00:00, 235.62it/s]


2026-07-06 11:03:03.946 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:03:03.947 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


convo_k4  (memory=Convo, memory_k=4, summary_k=0)
trial trcode  messages  summary
0     O_1     2         ''
1     O_2     4         ''
2     O_3     5         ''
3     C_1     5         ''
4     C_2     5         ''
5     C_3     5         ''


## D. Folding overflow into a summary — `summary_k`

Same `memory_k=4` window, but now `summary_k=2`: once the overflow (messages
pushed out of the window) reaches 2 messages, that overflow is summarized by
the model and folded into a rolling `summary` instead of being dropped
silently. The window size caps token growth the same way as (C); the
summary preserves what the window forgets.

In [6]:
_ = run_config("convo_k4_s2", memory="Convo", memory_k=4, summary_k=2)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_memory_quant_demo_runs/convo_k4_s2/example_survey/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [00:00, 265.83it/s]

2026-07-06 11:03:03.997 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:03:03.999 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


convo_k4_s2  (memory=Convo, memory_k=4, summary_k=2)
trial trcode  messages  summary
0     O_1     2         ''
1     O_2     4         ''
2     O_3     5         ''
3     C_1     5         'Summarize '
4     C_2     5         'Existing s'
5     C_3     5         'Existing s'


## Recap

| Config | `memory` | `memory_k` | `summary_k` | Message count by trial 5 | Summary produced? |
|---|---|---|---|---|---|
| A | `SingleTurn` | `-1` | `0` | 2 (always) | No |
| B | `Convo` | `-1` | `0` | 12 (unbounded) | No |
| C | `Convo` | `4` | `0` | 5 (capped) | No |
| D | `Convo` | `4` | `2` | 5 (capped) | Yes, once overflow ≥ `summary_k` |

`memory_k` controls token cost by capping how much raw history is replayed
each trial. `summary_k` is the knob for *not losing* what falls outside that
window — it trades exact wording for a compact paraphrase, folded into the
system message so thematic context survives long Convo runs.

## See also

- [Memory Types](../guides/memory_types.md) — conceptual overview of `memory` / `memory_k` / `summary_k`
- [Parameters Reference](02_parameters_reference.ipynb) — every other `ExpCard` field
- [`single_turn_convo.py`](../../src/psychscanner/memories/single_turn_convo.py) — `_trim_history` / `_make_summary` implementation